In [1]:
import pickle
import os ,re
import numpy as np
import torch
from torch.utils.data import DataLoader
# from avf import MyData
from dsac.data.angle_data import Data
from tqdm import tqdm

In [2]:

def bi(batch_labels , model_action):
    batch_labels = np.array(batch_labels)
    bi = 0
    for i in range(len(batch_labels)):
        if batch_labels[i] == model_action[i]:
            bi +=1 
    return bi/len(batch_labels)

In [3]:

from dsac import AVNet
path = ''
mydata = MyData(path=[path])
dataloader = DataLoader(dataset=mydata , batch_size=16 , shuffle= True)
model_path = './checkpoint/DSAC1.pth'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
agent = AVNet(128 , 4 ,width_dim=128 , height_dim=36).to(device)
agent.load_state_dict(torch.load(model_path))
agent.eval()
batch_data = next(iter(dataloader))
batch_pre_audio, batch_pre_visual, batch_next_audio , batch_next_visual,batch_done, batch_reward ,batch_labels = batch_data
batch_next_visual = batch_next_visual.to(device).float()
batch_next_audio = batch_next_audio.to(device).float()
batch_pre_audio = batch_pre_audio.to(device).float()
batch_pre_visual = batch_pre_visual.to(device).float()
model_action = agent(batch_pre_audio,batch_pre_visual)[2].max(-1)[1].tolist()
bi_list = list()
for batch_data in tqdm(dataloader ,desc='dataloder'):
    batch_pre_audio, batch_pre_visual, batch_next_audio , batch_next_visual,batch_done, batch_reward ,batch_labels = batch_data
    batch_next_visual = batch_next_visual.to(device).float()
    batch_next_audio = batch_next_audio.to(device).float()
    batch_pre_audio = batch_pre_audio.to(device).float()
    batch_pre_visual = batch_pre_visual.to(device).float()
    # model_action = agent(batch_pre_audio,batch_pre_visual).max(-1)[1].tolist()
    model_action = agent(batch_pre_audio,batch_pre_visual)[0].max(-1)[1].tolist()
    bi_list.append(bi(batch_labels=batch_labels , model_action=model_action))

ImportError: cannot import name 'AVNet' from 'dsac' (unknown location)

In [59]:
print(f"max_bi : {max(bi_list)},\nmin_bi : {min(bi_list)}, \nmean_bi :{np.array(bi_list).mean()}")

max_bi : 1.0,
min_bi : 0.5625, 
mean_bi :0.8705357142857143


In [39]:
# 测试foundation model
from avf import AVFNet
path = '../data/RL/best_action_forward_angle_valdata'
# path = '../data/RL/valdata'
files = [os.path.join(path  , i) for i in os.listdir(path)]
# print(files)
mydata = Data(path=files)
dataloader = DataLoader(dataset=mydata , batch_size=1 , shuffle= True)
model_path = '../data/checkpoint/acmcheckpoint/avf_38000.pth'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
agent =AVFNet(hid_dim=128 , out_put=4 ,width_dim=128 , height_dim=36).to(device)
agent.load_state_dict(torch.load(model_path))
agent.eval()

AVFNet(
  (audio): Sequential(
    (0): Conv2d(2, 32, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): Conv2d(32, 32, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(4, 4), stride=(1, 1), padding=(1, 1))
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (visual): Sequential(
    (0): Conv2d(4, 32, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): Conv2d(32, 32, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_m

In [36]:
batch_data = next(iter(dataloader))
batch_pre_audio, batch_pre_visual, batch_next_audio , batch_next_visual,batch_done, batch_reward ,batch_labels = batch_data
batch_next_visual = torch.stack(batch_next_visual).to(device).float()
batch_next_audio = torch.stack(batch_next_audio).to(device).float()
batch_pre_audio = torch.stack(batch_pre_audio).to(device).float()
batch_pre_visual = torch.stack(batch_pre_visual).to(device).float()
model_action = agent(batch_pre_audio,batch_pre_visual).max(-1)[1].tolist()

In [24]:
np.array(model_action)

array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0,
       0, 0, 2, 0, 0, 0, 0, 2, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0,
       0, 1])

In [25]:
torch.stack(batch_labels)

tensor([[1],
        [1],
        [1],
        [1],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [1],
        [0],
        [2],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [2],
        [0],
        [0],
        [3]])

In [40]:
bi_list = list()
for batch_data in tqdm(dataloader ,desc='dataloder'):
    batch_pre_audio, batch_pre_visual, batch_next_audio , batch_next_visual,batch_done, batch_reward ,batch_labels = batch_data
    batch_next_visual = torch.stack(batch_next_visual).to(device).float()
    batch_next_audio = torch.stack(batch_next_audio).to(device).float()
    batch_pre_audio = torch.stack(batch_pre_audio).to(device).float()
    batch_pre_visual = torch.stack(batch_pre_visual).to(device).float()
    # model_action = agent(batch_pre_audio,batch_pre_visual).max(-1)[1].tolist()
    model_action = agent(batch_pre_audio,batch_pre_visual).max(-1)[1].tolist()
    bi_list.append(bi(batch_labels=batch_labels , model_action=model_action))

dataloder: 100%|██████████| 100/100 [00:33<00:00,  3.03it/s]


In [41]:
print(f"max_bi : {max(bi_list)},\nmin_bi : {min(bi_list)}, \nmean_bi :{np.array(bi_list).mean()}")

max_bi : 0.9672131147540983,
min_bi : 0.3076923076923077, 
mean_bi :0.7148932715603676


In [29]:
a = torch.rand((4,4))
print(a)

tensor([[0.0641, 0.7972, 0.6782, 0.7989],
        [0.9255, 0.0740, 0.5069, 0.1337],
        [0.9480, 0.2002, 0.0207, 0.3708],
        [0.4686, 0.3238, 0.3769, 0.0065]])


In [31]:
a.max(dim = -1)[1]

tensor([3, 0, 0, 0])